# Load a pretrained decoder and keep training it

**Version.** This notebook needs Dew at or after the merge of the `hf-decoders` branch, which adds `dew.interop.load_pretrained_decoder` and `save_pretrained_decoder`. On a checkout from before that merge, the import in the loading cell raises.

You will load Qwen3-0.6B from Hugging Face into the same `CausalTransformer` the previous notebooks trained from scratch, generate text greedily from it, continue its pretraining for a few steps on Tiny Shakespeare with the regular `LMObjective`, and export the result back into the Hugging Face layout, where `transformers` loads it again. No weights are renamed by hand anywhere. The decoder's parameter tree already carries the Hugging Face names, so a load or a save is a translation of names, transposes and config fields.

Continued pretraining on 1 MB of Shakespeare at this learning rate changes the model very little. What the notebook shows is the round trip: load, train, export, reload.

**Expected time.** About 15 minutes on an A100, 20 on a TPU v5e and 20 to 30 on an RTX 4080, dominated by the one-off download of the 1.2 GB checkpoint and XLA compilation of a 0.6B model.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml @ git+https://github.com/AshishKumar4/dew" {jax_spec}
    %pip install -q transformers

In [ ]:
# Every size knob in one place.
CHECKPOINT = "Qwen/Qwen3-0.6B"
STEPS = 30           # continued-pretraining steps at batch 1, as the branch measured
BATCH_SIZE = 1
SEQUENCE_LENGTH = 512
LEARNING_RATE = 1e-5 # a fraction of pretraining: nudge, do not overwrite
MAX_NEW_TOKENS = 160
PROMPT = "To be, or not to be"
DATA_DIR = "shakespeare-qwen3"
OUT_DIR = "qwen3-shakespeare"

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## Load the checkpoint

`load_pretrained_decoder` takes a hub repo id or a local directory in the Hugging Face layout, downloads only the safetensors and json files, translates the config into `CausalTransformer` fields and the weights onto its parameter tree, and returns the model, its variables, and the dew config the translation produced. The parameters arrive in fp32 and the model computes in bf16, as in a from-scratch run, because `dtype` sets the compute dtype and leaves the stored parameters alone.

A config field that changes what the model computes and has no counterpart in Dew raises an error naming the field rather than being dropped. A tied `lm_head` is checked against the embedding it claims to copy and then dropped, because the tree has one leaf for the two. Parity against the reference implementation is the acceptance bar for a family, and Qwen3 passes it with the same argmax at every prompt position and a 1.4e-04 maximum logit difference on the real 0.6B weights.

In [ ]:
from dew.interop import load_pretrained_decoder

# The third value is the dew model config the checkpoint translated into, which
# is also what a --pretrained run logs.
model, variables, model_config = load_pretrained_decoder(CHECKPOINT, max_seq_len=1024)
print(f"{sum(x.size for x in jax.tree_util.tree_leaves(variables)) / 1e9:.2f}B parameters")
print({k: model_config[k] for k in
       ("vocab_size", "emb_features", "num_layers", "num_heads", "num_kv_heads",
        "head_dim", "qk_norm", "tie_embeddings", "max_seq_len")})

## Greedy generation, before training

`generate` is the same sampler the from-scratch notebook used. It prefills the prompt into the KV cache, then decodes in one `lax.scan`. With `temperature=0` it is greedy, so the output is a property of the weights alone and the same prompt will give exactly this text again after the round trip, which is the check at the end of the notebook.

The tokenizer is the checkpoint's own, because the model was trained on those ids. `dew.data.text.HFTokenizer` wraps any Hugging Face tokenizer behind the same `encode`/`decode` the byte tokenizer had.

In [ ]:
import jax.numpy as jnp
from dew.data.text import HFTokenizer
from dew.sampling.text import generate

tokenizer = HFTokenizer(CHECKPOINT)
prompt = jnp.asarray([tokenizer.encode(PROMPT)], jnp.int32)
before = generate(model, variables, prompt, max_new_tokens=MAX_NEW_TOKENS,
                  rng=jax.random.PRNGKey(0), temperature=0.0)
print(tokenizer.decode(before[0]))

## Continued pretraining

The token files are written the same way as before, with the checkpoint's tokenizer. `LMObjective` takes the pretrained `variables` through its `pretrained` argument, which is where `init_params` starts from instead of a fresh init. The trainer's whole initial state, optimizer included, is built on top of the loaded weights.

Thirty steps at batch 1 with a learning rate of 1e-5 barely moves the weights. The loss starts near 4, where a 0.6B model lands on Shakespeare in the Qwen tokenizer, and stays about there.

In [ ]:
import json
import urllib.request
from pathlib import Path

import numpy as np

DATA_DIR = Path(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw = DATA_DIR / "shakespeare.txt"
if not raw.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        raw)

ids = np.asarray(tokenizer.encode(raw.read_text(encoding="utf-8")))
val_len = int(round(len(ids) * 0.02))
val, train = ids[:val_len], ids[val_len:]
# 50257... the Qwen3 vocabulary needs more than 16 bits.
dtype = np.dtype("uint32")
val.astype(dtype).tofile(DATA_DIR / "val.bin")
train.astype(dtype).tofile(DATA_DIR / "train.bin")
meta = {"tokenizer": CHECKPOINT, "vocab_size": tokenizer.vocab_size, "dtype": dtype.name,
        "train_tokens": len(train), "val_tokens": len(val)}
(DATA_DIR / "meta.json").write_text(json.dumps(meta))
print(meta)

In [ ]:
from dew.data.dataloaders import get_token_dataset_grain
from dew.objectives.lm import LMObjective

data = get_token_dataset_grain(
    str(DATA_DIR / "train.bin"), str(DATA_DIR / "val.bin"),
    batch_size=BATCH_SIZE, seq_len=SEQUENCE_LENGTH, worker_count=2)

objective = LMObjective(
    model, SEQUENCE_LENGTH, vocab_size=meta["vocab_size"],
    pretrained=variables,
    samples={
        "prompt": tokenizer.encode(PROMPT),
        "max_new_tokens": 96,
        "temperature": 0.0,
        "decode": tokenizer.decode,
    })

In [ ]:
import optax
from dew.training import ObjectiveTrainer

trainer = ObjectiveTrainer(
    model, optax.adamw(LEARNING_RATE), objective=objective, input_config=None,
    rngs=jax.random.PRNGKey(0), name="qwen3-continued",
    checkpoint_base_path="./checkpoints")
state = trainer.fit(data, training_steps_per_epoch=STEPS, epochs=1,
                    val_steps_per_epoch=2)

## Export back, and reload in transformers

`save_pretrained_decoder` writes `config.json`, `model.safetensors` and `generation_config.json` in the Hugging Face vocabulary, using the same field map as the load, run backwards. `model_type` comes back out as `qwen3` here, from the presence of the q/k norms, and `transformers` loads the result as any other Qwen3 checkpoint.

The last cell reloads the export in `transformers` and takes the next token for the same prompt, if torch is installed on the runtime. The two frameworks picking the same token from the same weights is what closes the round trip.

In [ ]:
from dew.interop import save_pretrained_decoder

save_pretrained_decoder(model, state.params, OUT_DIR, tokenizer_name=CHECKPOINT)
print("wrote", OUT_DIR)

In [ ]:
try:
    import torch
except ModuleNotFoundError:
    print("torch is not installed on this runtime; install it to run the reload check")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    hf_model = AutoModelForCausalLM.from_pretrained(OUT_DIR, dtype=torch.float32)
    hf_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
    inputs = hf_tokenizer(PROMPT, return_tensors="pt")
    with torch.no_grad():
        logits = hf_model(**inputs).logits[0, -1]
    # The next token from transformers, against the one Dew produced above.
    print("transformers next token:", hf_tokenizer.decode(logits.argmax()))

## Where to go next

What loads today is `llama`, `qwen3` and `gemma3_text`, each gated by a parity test on real or random-weight fixtures; `qwen2` is refused by name, because it biases q, k and v and leaves `o_proj` bias-free, which the decoder's single `attention_bias` flag cannot say. The recipe flag `--pretrained` runs this notebook's flow at scale: `recipes/lm/train.py --pretrained Qwen/Qwen3-0.6B` continues any of these checkpoints on your own corpus, with the trainer, sharding and checkpoints of every other Dew run.